# GEHA generator LoRA finetuning on a Colab GPU

LoRA instruction-finetunes `Qwen/Qwen2.5-1.5B-Instruct` on GEHA dental RAG answers (direct + chain-of-thought formats), logs loss every step, plots the curves and compares base vs finetuned on the held-out plan brochure.

1. **Runtime → Change runtime type → GPU.** L4 or A100 train in bf16; a T4 works too (fp32, slower).
2. On your Mac, build the upload bundle: `cd /Users/dc/geha/finetuning_LLM_LORA && ./make_colab_zip.sh` → `dist/finetuning_LLM_LORA.zip`.
3. Run the cells in order. The last cell downloads `results/` and the adapter.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Upload dist/finetuning_LLM_LORA.zip from your Mac.
import zipfile
from google.colab import files

uploaded = files.upload()
zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
%cd /content/finetuning_LLM_LORA

In [ ]:
# Colab already ships torch with CUDA; only add the libraries it lacks.
!pip install -q "transformers>=5.0,<6" "peft>=0.21,<1" "accelerate>=1.1,<2" "matplotlib>=3.9,<4"
# Colab preinstalls torchao 0.10; peft rejects torchao < 0.16 when present. Unused here.
!pip uninstall -y -q torchao
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__)
print(torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Unit tests, SFT data build, split report + leakage checks.
!python -m unittest discover -s tests -t .
!python prepare_sft.py
!python split_stats.py --tokenizer

In [ ]:
# Train. Logs every optimizer step + validation every 10 steps to results/train_log.jsonl
# and keeps the lowest-validation-loss adapter in models/geha-dental-generator-lora.
EPOCHS = 3
!python train_lora.py --epochs {EPOCHS} --eval-every 5

In [ ]:
!python plot_loss.py
from IPython.display import Image, display
display(Image("results/loss_curve.png"))

In [ ]:
# Base vs finetuned on the held-out plan brochure (82 examples, greedy decoding).
!python eval_generator.py

In [ ]:
import json
import pandas as pd

comparison = json.load(open("results/generator_comparison.json"))
rows = [
    {"task": task, "metric": metric, **values}
    for task, metrics in comparison["metrics"].items()
    for metric, values in metrics.items()
    if metric != "n"
]
pd.DataFrame(rows).round(3)

In [ ]:
# Download results/ and the adapter back to the Mac.
!zip -qr /content/colab_results.zip results models
files.download("/content/colab_results.zip")